# Tüdőrák predikciója vizelet LC–MS metabolomikai profilból

### feature selection és gépi tanulási modellek összehasonlítása

### Ambrus Csaba


## 0. Setup

In [ ]:
# Init gdrive and python environment

from google.colab import drive
drive.mount('/content/drive')

%cd /content

import os
if not os.path.exists("/content/MetabolKD/.git"):
    !git clone https://github.com/csambrus/MetabolKD.git
else:
    %cd /content/MetabolKD
    !git fetch origin
    !git pull origin main

%cd /content/MetabolKD
!pip install -r ./requirements.txt

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = "content/MetabolKD"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
from src.runtime_setup import setup_tensorflow_runtime, setup_notebook_error_logger, set_global_seed

set_global_seed()
setup_notebook_error_logger()
setup_tensorflow_runtime()


## 1. Adatbetöltés


In [ ]:
from src.download_dataset import download_dataset, combine_pos_neg_to_feature_matrix

data = download_dataset()
X, meta = combine_pos_neg_to_feature_matrix(data)

# ML célváltozó
target_col = "class_label"  # Lung cancer vs Control

# Csak címkézett minták
mask = meta[target_col].notna()
X = X.loc[mask].copy()
meta = meta.loc[mask].copy()

# Biztonság: indexek egyezzenek
meta = meta.reindex(X.index)

X_raw = X
y = meta[target_col]

print("X_raw:", X_raw.shape)
print(y.value_counts(dropna=False))

## 2. QC és preprocessing

In [ ]:
from src.metabolomics_qc import qc_sample_table
from src.metabolomics_plotting import plot_qc_bars

qc = qc_sample_table(X_raw)
print(qc.describe())
fig = plot_qc_bars(qc, meta, hue_col="class_label")
fig.savefig("qc_overview_mtbls28.png", dpi=150, bbox_inches="tight")


### Preprocessing

In [ ]:
from src.metabolomics_preprocessing import nmr_style_matrices, preprocess_feature_matrix

blocks = nmr_style_matrices(X_raw, impute_first=True)
for k, df in blocks.items():
    print(k, df.shape)
X_pareto = blocks["X_pareto"]
X_log = blocks["X_log"]

X_proc, steps = preprocess_feature_matrix(X_raw)
print("Klasszikus LC–MS lépések:", " → ".join(steps))
X_proc.head()


## 3. PCA és explroatív adatstruktúra


In [ ]:
from src.metabolomics_multivariate import fit_pca
from src.metabolomics_plotting import plot_pca_multipanel, plot_pca_scores

pca_out = fit_pca(X_pareto, n_components=5, standardize=False)
scores = pca_out["scores"]
var = pca_out["explained_variance_ratio"]
print("Magyarázott variancia (első 5 PC):", [round(float(v), 4) for v in var])

fig = plot_pca_multipanel(pca_out, meta, hue_col="class_label")
fig.savefig("pca_multipanel_mtbls28.png", dpi=150, bbox_inches="tight")

fig2 = plot_pca_scores(
    scores, meta,
    pc_x="PC1", pc_y="PC2",
    hue_col="class_label",
    explained=var,
    title="PCA score (Pareto-skálázott log1p, MTBLS28)",
)
fig2.savefig("pca_scores_mtbls28.png", dpi=150, bbox_inches="tight")


## 4. Exploratív feature selection és biomarkerjelölt-keresés
### 4.1 Univariáns elemzés, FDR, volcano plot

In [ ]:
# MTBLS28: minden minta egy vizeletes profil; csoport = class_label
metab = meta

from src.metabolomics_univariate import differential_analysis
from src.metabolomics_plotting import plot_volcano, plot_volcano_categorized

diff = differential_analysis(
    X_log,
    metab["class_label"],
    group_a="Lung cancer",
    group_b="Control",
    include_cohen_d=True,
)
print(diff.head(15))
fig = plot_volcano(diff, alpha=0.05, fc_thresh=0.5)
fig.savefig("volcano_lung_cancer_vs_control_mtbls28.png", dpi=150, bbox_inches="tight")
figc = plot_volcano_categorized(
    diff,
    group_up="Lung cancer",
    group_down="Control",
    alpha=0.05,
    fc_thresh=0.5,
)
figc.savefig("volcano_categorized_mtbls28.png", dpi=150, bbox_inches="tight")



### 4.2 Top feature heatmap


In [ ]:
from src.metabolomics_plotting import plot_top_features_heatmap

top_n = 25
top_feats = diff.nsmallest(top_n, "padj")["feature"].tolist()
fig = plot_top_features_heatmap(
    X_pareto,
    top_feats,
    metab,
    group_col="class_label",
    max_samples=50,
)
fig.savefig("heatmap_top_features_mtbls28.png", dpi=150, bbox_inches="tight")


### 4.3 PLS-DA, VIP és S-plot


In [ ]:
from src.metabolomics_multivariate import fit_plsda_multiclass
from src.metabolomics_plotting import plot_s_plot_lv1
import matplotlib.pyplot as plt

plsda = fit_plsda_multiclass(
    X_pareto,
    metab["class_label"],
    n_components=3,
    scale=True,
)
s_pls = plsda["scores"]
vip = plsda["vip"]
print(vip.nlargest(10))

fig, ax = plt.subplots(figsize=(7, 5))
for lab in metab["class_label"].dropna().unique():
    m = metab["class_label"] == lab
    ax.scatter(s_pls.loc[m, "LV1"], s_pls.loc[m, "LV2"], label=str(lab), s=45, alpha=0.85, edgecolors="white", linewidths=0.3)
ax.set_xlabel("LV1")
ax.set_ylabel("LV2")
ax.set_title("PLS-DA score (MTBLS28: class_label)")
ax.legend(title="class_label")
ax.axhline(0, color="gray", lw=0.4)
ax.axvline(0, color="gray", lw=0.4)
fig.tight_layout()
fig.savefig("plsda_scores_mtbls28.png", dpi=150, bbox_inches="tight")

w1 = plsda["model"].pls.x_weights_[:, 0]
figs = plot_s_plot_lv1(X_pareto, metab["class_label"], "Lung cancer", w1)
figs.savefig("splot_lv1_mtbls28.png", dpi=150, bbox_inches="tight")


### 4.4 Random Forest feature importance

In [ ]:
from src.metabolomics_ml import random_forest_feature_importance

imp = random_forest_feature_importance(
    X_pareto,
    metab["class_label"],
    "Lung cancer",
    n_estimators=300,
)
print(imp.head(20))


## 5. Validált ML pipeline-ok: preprocessing, feature selection és modell-összehasonlítás
### 5.1 Feature selection stratégiák
### 5.2 Lasso és Ridge logisztikus regresszió
### 5.3 Random Forest
### 5.4 XGBoost
### 5.5 Neural Network / MLP
### 5.6 Modell-összehasonlítás

## 6. Modellbenchmark:
   ### Lasso / L1 logistic regression
   ### Ridge / L2 logistic regression
   ### Random Forest
   ### XGBoost
   ### Neural Network / MLP

## 7. Modell-összehasonlítás
   ### ROC-AUC
   ### F1
   ### balanced accuracy
   ### sensitivity / specificity
   ### confusion matrix